In [1]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, classification_report, confusion_matrix, precision_score
import numpy as np

In [2]:
df = pd.read_csv('foods_health_scores_allergens.csv')
df

,product_name,brands,categories,ingredients,nutriscore_grade,nova_group,ecoscore_grade,allergens,energy_kcal,fat_100g,...,proteins_100g,salt_100g,sodium_100g,contains_gluten,contains_dairy,contains_nuts,contains_soy,contains_eggs,contains_fish,food_type
0,Sidi Ali,سيدي علي,"en:beverages-and-beverages-preparations, en:be...",OBD1 999 999 1112606 266963207 mb,A,NaN,NOT-APPLICABLE,NaN,0.0,0.0,...,0.0,0.000000,0.000000,False,False,False,False,False,False,Branded/Packaged
1,Perly,Perly,"en:dairies, en:fermented-foods, en:fermented-m...","milk cream, cream, sugar, banana, bacteria",UNKNOWN,3.0,B,"en:banana, en:milk",97.0,3.0,...,8.0,NaN,NaN,False,True,False,False,False,False,Branded/Packaged
2,Sidi Ali,Sidi Ali,"en:beverages-and-beverages-preparations, en:be...","Sodium, Calcium, Magnésium, Potassium, Bicarbo...",A,1.0,NOT-APPLICABLE,NaN,NaN,NaN,...,NaN,0.065000,0.026000,False,False,False,False,False,False,Branded/Packaged
3,Eau minérale naturelle,sidi ali,"en:beverages-and-beverages-preparations, en:be...",100% mineral water,A,1.0,NOT-APPLICABLE,NaN,NaN,NaN,...,NaN,0.065000,0.026000,False,False,False,False,False,False,Branded/Packaged
4,اكوافينا,AQUAFINA,"en:beverages-and-beverages-preparations, en:be...",ouverture et avant le : Voir bouteille. après ...,A,NaN,NOT-APPLICABLE,NaN,0.0,0.0,...,0.0,0.000508,0.000203,False,False,False,False,False,False,Branded/Packaged
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4992,Crème fraîche gastronomique,Président,"en:dairies, en:fermented-foods, en:fermented-m...","_Crème_ (origine France), _ferments lactiques_",D,3.0,A,en:milk,291.0,30.0,...,2.3,0.070000,0.028000,False,True,False,False,False,False,Branded/Packaged
4993,Noix de cajou grillées sans sel,Maître Prunille,"en:plant-based-foods-and-beverages, en:plant-b...",Noix de cajou,B,1.0,E,en:nuts,621.0,47.0,...,21.0,0.020000,0.008000,False,False,True,False,False,False,Branded/Packaged
4994,Cacao puro en polvo desgrasado,Valor,"en:cocoa-and-its-products, en:cocoa-and-chocol...","Cacao desgrasado en polvo, correctores de acid...",C,1.0,F,NaN,375.0,16.0,...,26.0,0.030000,0.012000,False,False,False,False,False,False,Branded/Packaged
4995,Sablés Myrtilles Germes de riz,Gerblé,"en:snacks, en:sweet-snacks, en:biscuits-and-cakes","Farine de blé 58,2%, huile de colza, sucre de ...",A,4.0,C,"en:gluten, en:milk",54.0,2.0,...,0.9,0.050000,0.020000,True,True,False,False,False,False,Branded/Packaged


In [3]:
df["food_type"].value_counts()

food_type
Branded/Packaged    4997
Name: count, dtype: int64

In [4]:
df = df.drop(['product_name', 'brands', 'categories', 'ingredients', 'allergens', 'food_type'], axis = 1)

In [5]:
cols = [
    'contains_gluten',
    'contains_dairy',
    'contains_nuts',
    'contains_soy',
    'contains_eggs',
    'contains_fish'
]

## Model Pipeline Definition

In [6]:
# show numeric and categorical columns
num_cols = ['energy_kcal','fat_100g', 'saturated_fat_100g', 'carbs_100g', 'sugars_100g','fiber_100g', 'proteins_100g', 'salt_100g', 'sodium_100g']     
cat_cols = ['nutriscore_grade', 'nova_group', 'ecoscore_grade']     

# pipeline for numeric
num_pipeline = Pipeline([
    ("imputer", KNNImputer(n_neighbors=5)),
    ("scaler", StandardScaler())
])

# pipeline for categorical
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# combine preprocessing
preprocess = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

# full pipeline
pipe = Pipeline([
    ("preprocess", preprocess),
    ("lda", LinearDiscriminantAnalysis())
])

## Function Definitions

In [7]:
# Define thresholds
thresholds = np.arange(0.05, 0.96, 0.05)

# Drop unnecessary columns and perform train-test-validation split
def process(df, y_col: str, cols):
    y = df[y_col].astype(int)
    X = df.drop(cols, axis=1)

    X_trainval, X_test, y_trainval, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=5831,
        stratify=y
    )

    X_train, X_val, y_train, y_val = train_test_split(
        X_trainval, y_trainval,
        test_size=0.25,
        random_state=5831,
        stratify=y_trainval
    )

    return X_train, X_val, X_test, y_train, y_val, y_test

# Get recalls for each threshold
def get_recalls(thresholds, y_prob, y_val):
    recalls = dict()
    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        if precision_score(y_val, y_pred, pos_label=1) > 0.5:
            recalls[t] = recall_score(y_val, y_pred, pos_label = 1)
        
    max_key = max(recalls, key=recalls.get)
    return recalls, max_key

# Evaluate on the test set
def evaluate_on_test_set(pipe, X_test, y_test, max_key):
    y_prob = pipe.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= max_key)
    
    recall = recall_score(y_test, y_pred, pos_label=1)
    precision = precision_score(y_test, y_pred, pos_label=1)
    
    cm = confusion_matrix(y_test, y_pred)

    cm_df = pd.DataFrame(
        cm,
        index=["Actual 0", "Actual 1"],
        columns=["Pred 0", "Pred 1"]
    )

    return precision, recall, cm_df

# Get the DataFrame of model coefficients
def get_coef_df(pipe):
    feature_names = pipe.named_steps["preprocess"].get_feature_names_out()
    coefs = pipe.named_steps["lda"].coef_[0]

    coef_df = pd.DataFrame({
        "feature": feature_names,
        "coefficient": coefs,
        "abs_coefficient": np.abs(coefs)
    }).sort_values("abs_coefficient", ascending=False)

    return coef_df

## Analysis

### Gluten

In [8]:
%%script false --no-raise-error
gluten_X_train, gluten_X_val, gluten_X_test, gluten_y_train, gluten_y_val, gluten_y_test = process(df, 'contains_gluten', cols)

baseline_recall_gluten = (gluten_y_test == 1).mean()
print("Baseline recall for gluten:", baseline_recall_gluten)

In [9]:
%%script false --no-raise-error
pipe.fit(gluten_X_train, gluten_y_train)
gluten_y_prob = pipe.predict_proba(gluten_X_val)[:, 1]

In [10]:
%%script false --no-raise-error
gluten_recalls, gluten_max_key = get_recalls(thresholds, gluten_y_prob, gluten_y_val)
print(gluten_max_key)
print(gluten_recalls[gluten_max_key])

In [11]:
%%script false --no-raise-error
gluten_test_precision, gluten_test_recall, gluten_cm_df = evaluate_on_test_set(pipe, gluten_X_test, gluten_y_test, gluten_max_key)
print("Precision:", gluten_test_precision)
print("Recall:", gluten_test_recall)

In [12]:
%%script false --no-raise-error
print(gluten_cm_df)

In [13]:
%%script false --no-raise-error
gluten_coef_df = get_coef_df(pipe)
print(gluten_coef_df)

### Dairy

In [14]:
%%script false --no-raise-error
dairy_X_train, dairy_X_val, dairy_X_test, dairy_y_train, dairy_y_val, dairy_y_test = process(df, 'contains_dairy', cols)

baseline_recall_dairy = (dairy_y_test == 1).mean()
print("Baseline recall for dairy:", baseline_recall_dairy)

In [15]:
%%script false --no-raise-error
pipe.fit(dairy_X_train, dairy_y_train)
dairy_y_prob = pipe.predict_proba(dairy_X_val)[:, 1]

In [16]:
%%script false --no-raise-error
dairy_recalls, dairy_max_key = get_recalls(thresholds, dairy_y_prob, dairy_y_val)
print(dairy_max_key)
print(dairy_recalls[dairy_max_key])

In [17]:
%%script false --no-raise-error
dairy_test_precision, dairy_test_recall, dairy_cm_df = evaluate_on_test_set(pipe, dairy_X_test, dairy_y_test, dairy_max_key)
print("Precision for dairy:", dairy_test_precision)
print("Recall for dairy:", dairy_test_recall)

In [18]:
%%script false --no-raise-error
print(dairy_cm_df)

In [19]:
%%script false --no-raise-error
dairy_coef_df = get_coef_df(pipe)
print(dairy_coef_df)

### Nuts

In [20]:
%%script false --no-raise-error
nuts_X_train, nuts_X_val, nuts_X_test, nuts_y_train, nuts_y_val, nuts_y_test = process(df, 'contains_nuts', cols)

baseline_recall_nuts = (nuts_y_test == 1).mean()
print("Baseline recall for nuts:", baseline_recall_nuts)

Future exception was never retrieved
future: <Future finished exception=BrokenPipeError(32, 'Broken pipe')>
Traceback (most recent call last):
  File "/home/neh8/anaconda3/envs/dmsp26/lib/python3.10/asyncio/unix_events.py", line 676, in write
    n = os.write(self._fileno, data)
BrokenPipeError: [Errno 32] Broken pipe


In [21]:
%%script false --no-raise-error
pipe.fit(nuts_X_train, nuts_y_train)
nuts_y_prob = pipe.predict_proba(nuts_X_val)[:, 1]

In [22]:
%%script false --no-raise-error
nuts_recalls, nuts_max_key = get_recalls(thresholds, nuts_y_prob, nuts_y_val)
print(nuts_max_key)
print(nuts_recalls[nuts_max_key])

In [23]:
%%script false --no-raise-error
nuts_test_precision, nuts_test_recall, nuts_cm_df = evaluate_on_test_set(pipe, nuts_X_test, nuts_y_test, nuts_max_key)
print("Precision for nuts:", nuts_test_precision)
print("Recall for nuts:", nuts_test_recall)

In [24]:
%%script false --no-raise-error
print(nuts_cm_df)

In [25]:
%%script false --no-raise-error
nuts_coef_df = get_coef_df(pipe)
print(nuts_coef_df)

### Soy

In [26]:
%%script false --no-raise-error
soy_X_train, soy_X_val, soy_X_test, soy_y_train, soy_y_val, soy_y_test = process(df, 'contains_soy', cols)

baseline_recall_soy = (soy_y_test == 1).mean()
print("Baseline recall for soy:", baseline_recall_soy)

In [27]:
%%script false --no-raise-error
pipe.fit(soy_X_train, soy_y_train)
soy_y_prob = pipe.predict_proba(soy_X_val)[:, 1]

In [28]:
%%script false --no-raise-error
soy_recalls, soy_max_key = get_recalls(thresholds, soy_y_prob, soy_y_val)
print(soy_max_key)
print(soy_recalls[soy_max_key])

In [29]:
%%script false --no-raise-error
soy_test_precision, soy_test_recall, soy_cm_df = evaluate_on_test_set(pipe, soy_X_test, soy_y_test, soy_max_key)
print("Precision for soy:", soy_test_precision)
print("Recall for soy:", soy_test_recall)

In [30]:
%%script false --no-raise-error
print(soy_cm_df)

In [31]:
%%script false --no-raise-error
soy_coef_df = get_coef_df(pipe)
print(soy_coef_df)

### Eggs

In [32]:
%%script false --no-raise-error
eggs_X_train, eggs_X_val, eggs_X_test, eggs_y_train, eggs_y_val, eggs_y_test = process(df, 'contains_eggs', cols)

baseline_recall_eggs = (eggs_y_test == 1).mean()
print("Baseline recall for eggs:", baseline_recall_eggs)

In [33]:
%%script false --no-raise-error
pipe.fit(eggs_X_train, eggs_y_train)
eggs_y_prob = pipe.predict_proba(eggs_X_val)[:, 1]

In [34]:
%%script false --no-raise-error
eggs_recalls, eggs_max_key = get_recalls(thresholds, eggs_y_prob, eggs_y_val)
print(eggs_max_key)
print(eggs_recalls[eggs_max_key])

In [35]:
%%script false --no-raise-error
eggs_test_precision, eggs_test_recall, eggs_cm_df = evaluate_on_test_set(pipe, eggs_X_test, eggs_y_test, eggs_max_key)
print("Precision for eggs:", eggs_test_precision)
print("Recall for eggs:", eggs_test_recall)

In [36]:
%%script false --no-raise-error
print(eggs_cm_df)

In [37]:
%%script false --no-raise-error
eggs_coef_df = get_coef_df(pipe)
print(eggs_coef_df)

### Fish

In [38]:
%%script false --no-raise-error
fish_X_train, fish_X_val, fish_X_test, fish_y_train, fish_y_val, fish_y_test = process(df, 'contains_fish', cols)

baseline_recall_fish = (fish_y_test == 1).mean()
print("Baseline recall for fish:", baseline_recall_fish)

In [39]:
%%script false --no-raise-error
pipe.fit(fish_X_train, fish_y_train)
fish_y_prob = pipe.predict_proba(fish_X_val)[:, 1]

In [40]:
%%script false --no-raise-error
fish_recalls, fish_max_key = get_recalls(thresholds, fish_y_prob, fish_y_val)
print(fish_max_key)
print(fish_recalls[fish_max_key])

In [41]:
%%script false --no-raise-error
fish_test_precision, fish_test_recall, fish_cm_df = evaluate_on_test_set(pipe, fish_X_test, fish_y_test, fish_max_key)
print("Precision for fish:", fish_test_precision)
print("Recall for fish:", fish_test_recall)

In [42]:
%%script false --no-raise-error
print(fish_cm_df)

In [43]:
%%script false --no-raise-error
fish_coef_df = get_coef_df(pipe)
print(fish_coef_df)

We observe that our association analysis model has very low precision and recall for every variable except gluten. This could be partly due to a lack of samples containing the other allergens. We intend to tune our model hyperparameters to obtain sounder results since we are especially concerned about low recall because false negatives are more problematic than false positives when determining whether or not something contains an allergen.

## Including Other Allergy Columns

In [44]:
# Redefine cat_cols to include all columns
cat_cols = ['nutriscore_grade', 'nova_group', 'ecoscore_grade', 'contains_gluten', 'contains_dairy', 'contains_nuts', 'contains_soy', 'contains_eggs', 'contains_fish'] 

### Gluten

In [45]:
gluten_X_train, gluten_X_val, gluten_X_test, gluten_y_train, gluten_y_val, gluten_y_test = process(df, 'contains_gluten', 'contains_gluten')

baseline_recall_gluten = (gluten_y_test == 1).mean()
print("Baseline recall for gluten:", baseline_recall_gluten)

Baseline recall for gluten: 0.32


In [46]:
pipe.fit(gluten_X_train, gluten_y_train)
gluten_y_prob = pipe.predict_proba(gluten_X_val)[:, 1]

In [47]:
gluten_recalls, gluten_max_key = get_recalls(thresholds, gluten_y_prob, gluten_y_val)
print(gluten_max_key)
print(gluten_recalls[gluten_max_key])

0.15000000000000002
0.953125


In [48]:
gluten_test_precision, gluten_test_recall, gluten_cm_df = evaluate_on_test_set(pipe, gluten_X_test, gluten_y_test, gluten_max_key)
print("Precision:", gluten_test_precision)
print("Recall:", gluten_test_recall)

Precision: 0.5431192660550459
Recall: 0.925


In [49]:
print(gluten_cm_df)

          Pred 0  Pred 1
Actual 0     431     249
Actual 1      24     296


In [50]:
gluten_coef_df = get_coef_df(pipe)
print(gluten_coef_df)

                                 feature  coefficient  abs_coefficient
3                        num__carbs_100g     3.052845         3.052845
1                          num__fat_100g    -1.869161         1.869161
0                       num__energy_kcal     1.763504         1.763504
27    cat__ecoscore_grade_NOT-APPLICABLE    -1.700564         1.700564
15         cat__nutriscore_grade_UNKNOWN    -1.573583         1.573583
4                       num__sugars_100g    -1.435639         1.435639
14  cat__nutriscore_grade_NOT-APPLICABLE    -1.121382         1.121382
16                   cat__nova_group_1.0    -0.867836         0.867836
13               cat__nutriscore_grade_E     0.736893         0.736893
2                num__saturated_fat_100g    -0.682649         0.682649
19                   cat__nova_group_4.0     0.465574         0.465574
5                        num__fiber_100g    -0.388550         0.388550
6                     num__proteins_100g    -0.367780         0.367780
22    

### Dairy

In [51]:
# show numeric and categorical columns
num_cols = ['energy_kcal','fat_100g', 'saturated_fat_100g', 'carbs_100g', 'sugars_100g','fiber_100g', 'proteins_100g', 'salt_100g', 'sodium_100g']     
#cat_cols = ['nutriscore_grade', 'nova_group', 'ecoscore_grade']     

# pipeline for numeric
num_pipeline = Pipeline([
    ("imputer", KNNImputer(n_neighbors=5)),
    ("scaler", StandardScaler())
])

# pipeline for categorical
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# combine preprocessing
preprocess = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

# full pipeline
pipe = Pipeline([
    ("preprocess", preprocess),
    ("lda", LinearDiscriminantAnalysis())
])

In [52]:
dairy_X_train, dairy_X_val, dairy_X_test, dairy_y_train, dairy_y_val, dairy_y_test = process(df, 'contains_dairy', 'contains_dairy')

baseline_recall_dairy = (dairy_y_test == 1).mean()
print("Baseline recall for dairy:", baseline_recall_dairy)

Baseline recall for dairy: 0.299


In [53]:
pipe.fit(dairy_X_train, dairy_y_train)
dairy_y_prob = pipe.predict_proba(dairy_X_val)[:, 1]

ValueError: A given column is not a column of the dataframe

In [ ]:
dairy_recalls, dairy_max_key = get_recalls(thresholds, dairy_y_prob, dairy_y_val)
print(dairy_max_key)
print(dairy_recalls[dairy_max_key])

In [ ]:
dairy_test_precision, dairy_test_recall, dairy_cm_df = evaluate_on_test_set(pipe, dairy_X_test, dairy_y_test, dairy_max_key)
print("Precision for dairy:", dairy_test_precision)
print("Recall for dairy:", dairy_test_recall)

In [ ]:
print(dairy_cm_df)

In [ ]:
dairy_coef_df = get_coef_df(pipe)
print(dairy_coef_df)

### Nuts

In [ ]:
nuts_X_train, nuts_X_val, nuts_X_test, nuts_y_train, nuts_y_val, nuts_y_test = process(df, 'contains_nuts', 'contains_nuts')

baseline_recall_nuts = (nuts_y_test == 1).mean()
print("Baseline recall for nuts:", baseline_recall_nuts)

In [ ]:
pipe.fit(nuts_X_train, nuts_y_train)
nuts_y_prob = pipe.predict_proba(nuts_X_val)[:, 1]

In [ ]:
nuts_recalls, nuts_max_key = get_recalls(thresholds, nuts_y_prob, nuts_y_val)
print(nuts_max_key)
print(nuts_recalls[nuts_max_key])

In [ ]:
nuts_test_precision, nuts_test_recall, nuts_cm_df = evaluate_on_test_set(pipe, nuts_X_test, nuts_y_test, nuts_max_key)
print("Precision for nuts:", nuts_test_precision)
print("Recall for nuts:", nuts_test_recall)

In [ ]:
print(nuts_cm_df)

In [ ]:
nuts_coef_df = get_coef_df(pipe)
print(nuts_coef_df)

### Soy

In [ ]:
soy_X_train, soy_X_val, soy_X_test, soy_y_train, soy_y_val, soy_y_test = process(df, 'contains_soy', 'contains_soy')

baseline_recall_soy = (soy_y_test == 1).mean()
print("Baseline recall for soy:", baseline_recall_soy)

In [ ]:
pipe.fit(soy_X_train, soy_y_train)
soy_y_prob = pipe.predict_proba(soy_X_val)[:, 1]

In [ ]:
soy_recalls, soy_max_key = get_recalls(thresholds, soy_y_prob, soy_y_val)
print(soy_max_key)
print(soy_recalls[soy_max_key])

In [ ]:
soy_test_precision, soy_test_recall, soy_cm_df = evaluate_on_test_set(pipe, soy_X_test, soy_y_test, soy_max_key)
print("Precision for soy:", soy_test_precision)
print("Recall for soy:", soy_test_recall)

In [ ]:
print(soy_cm_df)

In [ ]:
soy_coef_df = get_coef_df(pipe)
print(soy_coef_df)

### Eggs

In [ ]:
eggs_X_train, eggs_X_val, eggs_X_test, eggs_y_train, eggs_y_val, eggs_y_test = process(df, 'contains_eggs', 'contains_eggs')

baseline_recall_eggs = (eggs_y_test == 1).mean()
print("Baseline recall for eggs:", baseline_recall_eggs)

In [ ]:
pipe.fit(eggs_X_train, eggs_y_train)
eggs_y_prob = pipe.predict_proba(eggs_X_val)[:, 1]

In [ ]:
eggs_recalls, eggs_max_key = get_recalls(thresholds, eggs_y_prob, eggs_y_val)
print(eggs_max_key)
print(eggs_recalls[eggs_max_key])

In [ ]:
eggs_test_precision, eggs_test_recall, eggs_cm_df = evaluate_on_test_set(pipe, eggs_X_test, eggs_y_test, eggs_max_key)
print("Precision for eggs:", eggs_test_precision)
print("Recall for eggs:", eggs_test_recall)

In [ ]:
print(eggs_cm_df)

In [ ]:
eggs_coef_df = get_coef_df(pipe)
print(eggs_coef_df)

### Fish

In [ ]:
fish_X_train, fish_X_val, fish_X_test, fish_y_train, fish_y_val, fish_y_test = process(df, 'contains_fish', 'contains_fish')

baseline_recall_fish = (fish_y_test == 1).mean()
print("Baseline recall for fish:", baseline_recall_fish)

In [ ]:
pipe.fit(fish_X_train, fish_y_train)
fish_y_prob = pipe.predict_proba(fish_X_val)[:, 1]

In [ ]:
fish_recalls, fish_max_key = get_recalls(thresholds, fish_y_prob, fish_y_val)
print(fish_max_key)
print(fish_recalls[fish_max_key])

In [ ]:
fish_test_precision, fish_test_recall, fish_cm_df = evaluate_on_test_set(pipe, fish_X_test, fish_y_test, fish_max_key)
print("Precision for fish:", fish_test_precision)
print("Recall for fish:", fish_test_recall)

In [ ]:
print(fish_cm_df)

In [ ]:
fish_coef_df = get_coef_df(pipe)
print(fish_coef_df)